# Provenance Agent — Software Workflow (Quickstart)

This notebook walks through the **software** side of the provenance agent. Given any Jupyter notebook, it identifies which Python libraries are imported and injects a cell that builds a DataFrame of their citation metadata.

Dataset citations are handled separately by the **data workflow** — see `data_workflow.ipynb`.

**Functions covered:**
- `parse_notebook` — reads a `.ipynb` and returns imported libraries
- `extract_libraries` — extracts imports from a single string
- `strip_ipython_directives` — removes magics/shell lines so code parses
- `validate_libraries` — checks which requested libraries exist
- `collect_library_entries` — merges the matching BibTeX into the citation-metadata DataFrame

## Setup

Add `src/` to the path so the module can be imported without installing the package.

In [ ]:
import numpy
import pandas

from provenance_agent.notebook_io import (
    parse_notebook,
    extract_libraries,
    strip_ipython_directives,
)

## 1. `parse_notebook` — scan a full notebook

Pass a path to any `.ipynb` file and it returns a dict whose `libraries` key is the sorted list of imported library names. (It also returns a `datasets` key used by the separate data workflow; the software workflow only reads `libraries`.)

In [ ]:
result = parse_notebook('../fixtures/sample.ipynb')
print('Libraries found:', result)

Automatically parses current notebook with nbformat

In [ ]:
result = parse_notebook()
print('Libraries found:', result)

You can also point it at any other notebook by passing an explicit path:

```python
result = parse_notebook('sample.ipynb')
```

## 2. `extract_libraries` — parse a snippet of code

`extract_libraries` works on a raw string rather than a file. It's useful for testing or for processing individual cells. It handles both `import X` and `from X.Y import Z` forms, and always returns the top-level package name (e.g. `matplotlib.pyplot` → `matplotlib`).

In [ ]:
code = """
import numpy as np
import pandas as pd
from matplotlib.pyplot import plt
"""
print(extract_libraries(code))

## 3. `strip_ipython_directives` — clean up magic and shell commands

Paleoclimate notebooks commonly use `%matplotlib inline`, `!pip install ...`, and similar IPython-only syntax. These lines are not valid Python and would cause `ast.parse` to fail. `strip_ipython_directives` removes them so that import extraction can proceed on the rest of the cell.

In [ ]:
raw = """
%matplotlib inline
!pip install pyleoclim
import pyleoclim as pyleo
"""
print(strip_ipython_directives(raw))

The `%` and `!` lines are dropped; the `import` line passes through unchanged. `parse_notebook` calls this automatically on every cell before parsing.

## 5. Full Pipeline Test — parse → validate → collect

Test the software citation pipeline:
1. `parse_notebook` — extract imported libraries
2. `validate_libraries` — check which requested libraries are available
3. `collect_library_entries` — merge their BibTeX from the packaged `Citations/` data into one DataFrame

There is no rendering step. APA rendering and the assembled-text bibliography were removed along with the LLM chain behind them. The citations *are* the DataFrame, and the workflow's job is to inject a cell that builds and displays it.

In [ ]:
from provenance_agent.notebook_io import validate_libraries
from provenance_agent.citations import collect_library_entries

In [ ]:
# Step 1: Parse the example notebook
result = parse_notebook('../examples/paleoPCAlite.ipynb')
print('All libraries found:', result)

In [ ]:
# Step 2: Validate a subset of requested libraries
found, not_found = validate_libraries(['pandas', 'numpy', 'fake_lib'], result)
print(f'Found: {found}')
print(f'Not found: {not_found}')

In [ ]:
# Step 3: Collect BibTeX entries
entries = collect_library_entries(found)
print(f'Collected {len(entries)} BibTeX entries')
entries

---
## Testing on Real Notebooks

The five notebooks below are from the *Comparing Simulated and Reconstructed Climate Variability over the Past Millennium* collection. Each one loads CMIP6/PMIP model output and LMR reconstruction data via `intake`, `xarray`, and `zarr`. Running `parse_notebook` on them exercises the parser against real scientific notebooks.

### C02_b — Data Assimilation with Individual Seasonality

This notebook uses `cfr` (Climate Field Reconstruction), `numpy`, and `pickle`. It exercises the parser on a paleoclimate-specific workflow that includes serialized data loading.

In [ ]:
result = parse_notebook('../examples/C02_b_DA_with_individual_seasonality.ipynb')
print(f'Libraries: {result}')
assert result == ['cfr', 'numpy', 'pickle', 'tqdm'], f'Unexpected: {result}'
print('PASSED')

In [ ]:
notebooks = [
    '../examples/comparing-simulated-reconstructed-climate/CMIP6_LMR.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/data_from_esm_cloudcat.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/spatial_snapshots_xarray_bonuses.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/VICS_dashboard.ipynb',
    '../examples/comparing-simulated-reconstructed-climate/widget_primer.ipynb',
]

for nb_path in notebooks:
    result = parse_notebook(nb_path)
    print(f'{nb_path.split("/")[-1]}')
    print(f'  Libraries: {result}')
    print()

## Full Pipeline Test on C02_b

Run the entire bibliography pipeline on a real paleoclimate notebook: parse → validate → collect → render (LLM) → generate.

In [ ]:
# Step 1: Parse
result = parse_notebook('../examples/paleoPCAlite.ipynb')
print('Libraries found:', result)

In [ ]:
# Step 2: Validate
found, not_found = validate_libraries(result, result)
print(f'Found: {found}')
print(f'Not found: {not_found}')

In [ ]:
# Step 3: Collect BibTeX entries
entries = collect_library_entries(found)
print(f'Collected {len(entries)} BibTeX entries')
entries

In [ ]:
entries

In [ ]:
# Step 4: Render to APA via Gemini
apa_text = render_apa(entries)
print(apa_text)

In [ ]:
# Step 5: Full bibliography
bib_text = generate_bibliography(result)
print(bib_text)

---

## 6. The agent layer - three ways to call it

Everything above calls the software workflow functions directly. The agent wraps that same work in three layers, shown here lowest to highest. They all produce the same software citations; they differ only in how you invoke them.

- **Direct tool** - `cite_software`, a plain function call.
- **NL router** - `agent.run`, where the model picks the tool and args from a plain-English request.
- **`%provenance` magic** - the same request as a notebook line magic.

### The direct tool - `cite_software`

`cite_software` is the function the other two layers ultimately call. Like `cite_data`, it does not return citation text - it **injects a cell** into the notebook. That one cell binds `provenance_software = collect_library_entries(...)` (parsing the packaged `.bib` files with bibtexparser) and `display()`s it. There is no combine cell: each workflow owns exactly one cell, and a re-run replaces its own rather than stacking a stale second one. It returns the list of libraries the cell was built for; the citations appear when you run that cell. Pass `output_path` to write to a copy instead of in place.

In [ ]:
from provenance_agent import cite_software

# cite_software injects one self-displaying metadata cell; write to a
# throwaway copy so the example notebook is not mutated in place.
libraries = cite_software(
    '../examples/paleoPCAlite.ipynb',
    output_path='../examples/paleoPCAlite_demo.ipynb',
)
print('injected a citation cell for:', libraries)

It also takes the filters the NL layers expose through language: `libraries=` to cite only specific packages, and `citation_types=` to keep only `paper` or `software` entries.

In [ ]:
libraries = cite_software(
    '../examples/paleoPCAlite.ipynb',
    libraries=['pyleoclim', 'pandas'],
    citation_types=['software'],
    output_path='../examples/paleoPCAlite_demo.ipynb',
)
print('injected a citation cell for:', libraries)

### The natural-language router - `agent.run`

`agent.run` classifies the request and dispatches the workflows it selected. It returns an envelope whose `dispatch` key holds one `{name, args, result}` dict per tool it called; for software the `result` is the list of libraries the injected cell was built for, and the citations themselves appear when you run that cell and see the `provenance_software` DataFrame. Note this injects **in place** into the notebook you point it at.

In [ ]:
from provenance_agent.agent import run

result = run('cite the software in BibTeX', '../examples/paleoPCAlite.ipynb')
for call in result['dispatch']:
    print(call['result'])

### The `%provenance` magic

The highest layer: one natural-language line in a notebook cell, and the agent picks the tool and arguments itself. The package is installed (`pip install -e ".[dev]"`), so there is nothing to add to `sys.path`.

Each tool **injects exactly one cell** into the target notebook - software: a metadata cell binding `provenance_software`; data: one retrieval cell covering every detected dataset. There is no combine cell; each workflow owns its own cell and a re-run replaces it. They report what they wrote rather than printing citations. Because injection is **in place**, the cells below point the magic at a throwaway copy of the example.

In [10]:
%load_ext provenance

The provenance extension is already loaded. To reload it, use:
  %reload_ext provenance


If auto-detection of the current notebook fails (common in VSCode - `ipynbname` matches the kernel against the Jupyter server's session list), set the path once for the session:

In [ ]:
import shutil
shutil.copy('software_workflow.ipynb', 'software_workflow_demo.ipynb')
%provenance_notebook software_workflow_demo.ipynb

### All software

Routes to `cite_software`, which injects one metadata cell into the target notebook and reports the libraries it covered. Reload the target and run that cell to see the `provenance_software` DataFrame.

In [ ]:
%provenance cite the software

### One library

The agent pulls `Pyleoclim` out of the request and passes it through as a filter.

In [ ]:
%provenance cite Pyleoclim

### Format

Software output is always the `provenance_software` DataFrame. There is no APA-vs-BibTeX choice anywhere anymore - APA rendering was removed, and `fmt` is accepted and ignored - so naming a format in the request changes nothing.

In [ ]:
%provenance cite the software in BibTeX

### Datasets

The same magic routes to the data workflow. Retrieval needs the dataset objects live in the kernel, so instead of returning citations this **injects one retrieval cell covering every detected dataset** and reports what it wrote. The citations are that cell's output once you reload and run it: each source's `_bib_{var}` and `_meta_{var}` stay bound in the kernel, and the cell displays every metadata frame.

It writes **in place**, so we point it at a throwaway copy of the example rather than the canonical one.

In [ ]:
import shutil
shutil.copy('../examples/paleoPCAlite.ipynb', '../examples/paleoPCAlite_demo.ipynb')
%provenance_notebook ../examples/paleoPCAlite_demo.ipynb
%provenance cite the datasets

In [ ]:
# provenance-agent-generated
from provenance_agent.citations import collect_library_entries
provenance_software = collect_library_entries(['numpy', 'pandas', 'provenance_agent'], None)
display(provenance_software)